# ARTEMIS — Pipeline RAG + Tool Calling
**Competencia MAPL-202601 | Grupo 2**

Secciones requeridas:
1. Data loading and exploration
2. Preprocessing
3. Model training (LoRA fine-tuning)
4. Inference (RAG pipeline)
5. Answer file generation

> **Entorno recomendado**: Secciones 1-2 + chunking en local (Mac).  
> Secciones 3-5 en **Colab Pro con A100** (~2.5 h de entrenamiento).

## 0 · Setup

In [1]:
# Instalar dependencias (descomentar en Colab)
# !pip install -q -r requirements.txt
# En Colab con GPU: usar faiss-gpu
# !pip install -q faiss-gpu

In [2]:
import json
import re
import unicodedata
from difflib import get_close_matches
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Rutas base — ajustar si se sube a Colab con Drive
BASE   = Path(".")          # raíz del repo (Sebastian/)
DATA   = BASE / "Data"
KB_DIR = DATA / "knowledge_base" / "knowledge_base"

RANDOM_STATE  = 42
RETRIEVAL_K   = 5   # chunks a recuperar de FAISS
PROMPT_K      = 3   # chunks que van al prompt
MAX_NEW_TOKS  = 80  # tokens máximos a generar

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

PyTorch 2.5.1 | CUDA: False


---
## Sección 1 — Data Loading & Exploration

In [3]:
# ── Cargar datos principales ──────────────────────────────────────────────────
train_raw = pd.read_csv(DATA / "train.csv")
test_df   = pd.read_csv(DATA / "test.csv")

print(f"Train: {len(train_raw)} filas | Test: {len(test_df)} filas")
train_raw.head(3)

Train: 2718 filas | Test: 766 filas


,id,query,tool_call
0,Q-00001,Cóndor radiation sensors are showing 2.1 mSv/h...,"activate_protocol(protocol_id='MASA-SEC-012',s..."
1,Q-00002,"We’ve got a rapid temperature rise in Quetzal,...","activate_protocol(protocol_id='MASA-SEC-003',s..."
2,Q-00003,Can we get the trajectory calculations for the...,"calculate_trajectory(maneuver='reentry',urgenc..."


In [4]:
# ── Cargar tools_definition.json ─────────────────────────────────────────────
with open(DATA / "tools_definition.json") as f:
    tools_def_raw = json.load(f)

# Convertir lista a dict {tool_name: {parameters: {...}}}
TOOLS_DEF = {t["name"]: t for t in tools_def_raw["tools"]}
VALID_TOOLS = list(TOOLS_DEF.keys())

print("Herramientas disponibles:")
for name, info in TOOLS_DEF.items():
    params = list(info.get("parameters", {}).keys())
    print(f"  {name}({', '.join(params)})")

Herramientas disponibles:
  get_telemetry(module, metric, timeframe_hours)
  get_crew_status(module, info)
  get_module_status(module, system)
  send_alert(module, severity, reason)
  send_message(recipient, priority)
  schedule_maintenance(module, task, priority)
  activate_protocol(protocol_id, scope)
  control_system(module, system, action)
  calculate_trajectory(maneuver, urgency)
  request_supply(category, urgency)
  no_action()


In [5]:
# ── Distribución de herramientas en train ────────────────────────────────────
def extract_tool_name(tc: str) -> str:
    return tc.split("(")[0].strip()

train_raw["tool_name"] = train_raw["tool_call"].apply(extract_tool_name)
dist = train_raw["tool_name"].value_counts()
print("Distribución de tools:")
print(dist.to_string())

Distribución de tools:
tool_name
activate_protocol       797
send_alert              675
no_action               331
control_system          162
calculate_trajectory    121
request_supply          121
schedule_maintenance    117
get_telemetry           107
get_module_status       104
get_crew_status          93
send_message             90


In [6]:
# ── Explorar knowledge base ───────────────────────────────────────────────────
kb_docs = sorted(KB_DIR.rglob("doc.md"))
print(f"Documentos en knowledge_base: {len(kb_docs)}")
for p in kb_docs[:5]:
    size = len(p.read_text(encoding="utf-8"))
    print(f"  {p.parent.name}: {size} chars")

Documentos en knowledge_base: 54
  MASA-DOC-001: 11692 chars
  MASA-DOC-002: 14041 chars
  MASA-DOC-003: 12551 chars
  MASA-DOC-004: 12710 chars
  MASA-DOC-005: 10708 chars


In [7]:
# ── Cargar consultas para métricas de retrieval ───────────────────────────────
with open(DATA / "consultas_centro_control.json") as f:
    consultas = json.load(f)

print(f"Consultas para evaluación de retrieval: {len(consultas)}")
has_hard_neg = sum(1 for q in consultas if "hard_negative_doc_id" in q)
print(f"Con hard negative: {has_hard_neg}")

Consultas para evaluación de retrieval: 810
Con hard negative: 622


---
## Sección 2 — Preprocessing

In [8]:
# ── 2.1 Eliminar duplicados exactos ──────────────────────────────────────────
df = train_raw.copy()
before = len(df)
df = df.drop_duplicates(subset=["query"])
print(f"Duplicados eliminados: {before - len(df)} (quedan {len(df)})")

Duplicados eliminados: 148 (quedan 2570)


In [9]:
# ── 2.2 Validar tool_calls — eliminar filas con herramienta inválida ──────────
before = len(df)
df = df[df["tool_name"].isin(VALID_TOOLS)].copy()
print(f"Filas con tool inválida eliminadas: {before - len(df)} (quedan {len(df)})")

Filas con tool inválida eliminadas: 0 (quedan 2570)


In [10]:
# ── 2.3 Normalizar queries (unicode NFC, strip, whitespace) ───────────────────
def normalize_query(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["query"] = df["query"].apply(normalize_query)
test_df["query"] = test_df["query"].apply(normalize_query)

print(f"Dataset limpio: {len(df)} ejemplos")
print("\nDistribución final:")
print(df["tool_name"].value_counts().to_string())

Dataset limpio: 2570 ejemplos

Distribución final:
tool_name
activate_protocol       771
send_alert              662
no_action               296
control_system          162
schedule_maintenance    117
request_supply          114
get_telemetry           107
get_module_status       102
send_message             82
calculate_trajectory     80
get_crew_status          77


In [11]:
# ── 2.4 Split 90/10 estratificado por herramienta ────────────────────────────
train_df, val_df = train_test_split(
    df, test_size=0.10, random_state=RANDOM_STATE, stratify=df["tool_name"]
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Validation: {len(val_df)}")

Train: 2313 | Validation: 257


---
## Sección 3 — Model Training
### 3.1 Chunking + Embedding + FAISS Index

In [12]:
import sys
sys.path.insert(0, str(BASE))
from scripts.chunker import SentenceTextSplitter, load_markdown_as_pages

splitter = SentenceTextSplitter(section_length=800, overlap_pct=0.20, max_tokens=400)

chunks = []
for doc_path in sorted(KB_DIR.rglob("doc.md")):
    doc_id = doc_path.parent.name
    pages  = load_markdown_as_pages(str(doc_path))
    for i, sp in enumerate(splitter.split_pages(pages)):
        chunks.append({"doc_id": doc_id, "chunk_id": i, "text": sp.text})

print(f"Total chunks: {len(chunks)}")

Total chunks: 785


In [ ]:
from sentence_transformers import SentenceTransformer

# device="cpu": evita cuelgues con MPS en Mac; en Colab detecta "cuda" automáticamente
_device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=_device)

texts  = [c["text"] for c in chunks]
vecs   = encoder.encode(
    texts,
    batch_size=16,               # conservador para CPU; subir a 64 en Colab GPU
    normalize_embeddings=True,   # cosine via dot product en FAISS
    show_progress_bar=True,
)

print(f"Embeddings shape: {vecs.shape}")

In [ ]:
# ── Construir índice FAISS ────────────────────────────────────────────────────
dim   = vecs.shape[1]   # 384 para bge-small
index = faiss.IndexFlatIP(dim)
index.add(vecs.astype(np.float32))

print(f"FAISS index: {index.ntotal} vectores | dim={dim}")

# Guardar índice para reutilizar sin re-embedear
faiss.write_index(index, str(BASE / "faiss.index"))

In [ ]:
# ── Guardar retrieval_index.json (entregable obligatorio) ─────────────────────
retrieval_index = []
for i, c in enumerate(chunks):
    retrieval_index.append({
        "doc_id":   c["doc_id"],
        "chunk_id": c["chunk_id"],
        "text":     c["text"],
        "vector":   vecs[i].tolist(),
    })

with open(BASE / "retrieval_index.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_index, f, ensure_ascii=False)

print("retrieval_index.json guardado.")

In [ ]:
# ── Función de retrieval ──────────────────────────────────────────────────────
def retrieve(query: str, k: int = RETRIEVAL_K) -> list:
    q_vec = encoder.encode([query], normalize_embeddings=True).astype(np.float32)
    _, idxs = index.search(q_vec, k)
    return [chunks[i] for i in idxs[0] if i < len(chunks)]

# Smoke test
sample = retrieve("Cóndor radiation emergency protocol")
print(f"Retrieval test — top-1 doc_id: {sample[0]['doc_id']}")

In [ ]:
# ── Métricas de retrieval P@K y R@K ─────────────────────────────────────────
def retrieval_metrics(k: int = 3) -> dict:
    hits = 0
    for item in consultas:
        q   = normalize_query(item["query"])
        rel = item["doc_id"]
        retrieved_ids = [c["doc_id"] for c in retrieve(q, k=k)]
        if rel in retrieved_ids:
            hits += 1
    score = hits / len(consultas)
    # Con 1 doc relevante por query: P@K = R@K = Recall@K
    return {f"P@{k}": round(score, 4), f"R@{k}": round(score, 4)}

for k in [1, 3, 5]:
    print(retrieval_metrics(k))

### 3.2 Generar contexto de retrieval para el conjunto de entrenamiento

Recuperamos los top chunks ANTES de fine-tunar para incluirlos como contexto en el prompt de entrenamiento.

In [ ]:
print("Generando contexto de retrieval para train...")
train_df["context_chunks"] = train_df["query"].apply(lambda q: retrieve(q, k=PROMPT_K))

print("Generando contexto de retrieval para validation...")
val_df["context_chunks"] = val_df["query"].apply(lambda q: retrieve(q, k=PROMPT_K))

print("Listo.")

### 3.3 LoRA Fine-tuning de Llama-3.2-1B-Instruct

> **Ejecutar en Colab con A100** (~2.5 h). En T4 (~5-7 h).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_cfg = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",         # MLP
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# ── System prompt y template ──────────────────────────────────────────────────
SYSTEM_PROMPT = """You are ARTEMIS, the AI control system for MASA's Kuntur Station.
Given an operator query and relevant documentation, output ONLY the exact tool call.

Format rules (STRICT — any deviation = wrong answer):
- No spaces after commas or around '=' signs
- Single quotes for string values: module='condor'
- Integer values without quotes: timeframe_hours=6
- Parameter ORDER must match the tool definition exactly
- Module names are lowercase ASCII: condor, quetzal, jaguar, colibri, vicuna, tucan
- Protocol IDs UPPERCASE: MASA-SEC-012
- For purely informational queries with no system action: no_action"""

def make_prompt(query: str, ctx_chunks: list) -> str:
    ctx = "\n\n".join(f"[{c['doc_id']}]\n{c['text']}" for c in ctx_chunks)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Query: {query}\n\nContext:\n{ctx}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
# ── Construcción del dataset ──────────────────────────────────────────────────
# Solo se entrena la respuesta (tool_call); el contexto se enmascara con -100.
# Mismo patrón que Microproyecto 3: solo los tokens de la respuesta correcta.
from torch.utils.data import Dataset as TorchDataset

MAX_SEQ = 768
LABEL_IGNORE = -100

class ToolCallDataset(TorchDataset):
    def __init__(self, df: pd.DataFrame):
        self.examples = []
        for _, row in df.iterrows():
            prompt   = make_prompt(row["query"], row["context_chunks"])
            full_str = prompt + row["tool_call"] + tokenizer.eos_token

            prompt_ids = tokenizer.encode(prompt,    add_special_tokens=False)
            full_ids   = tokenizer.encode(full_str,  add_special_tokens=False)

            if len(full_ids) > MAX_SEQ:
                full_ids   = full_ids[:MAX_SEQ]
                prompt_ids = prompt_ids[:MAX_SEQ]

            n_ctx  = min(len(prompt_ids), len(full_ids))
            labels = [LABEL_IGNORE] * n_ctx + full_ids[n_ctx:]

            # Sanity: al menos 1 token entrenado
            if len(labels) > n_ctx:
                self.examples.append({
                    "input_ids":      full_ids,
                    "attention_mask": [1] * len(full_ids),
                    "labels":         labels,
                })

    def __len__(self):  return len(self.examples)
    def __getitem__(self, i): return self.examples[i]


train_dataset = ToolCallDataset(train_df)
val_dataset   = ToolCallDataset(val_df)
print(f"Train examples: {len(train_dataset)} | Val examples: {len(val_dataset)}")

In [ ]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=LABEL_IGNORE,
    pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir=str(BASE / "decoder_checkpoint"),
    num_train_epochs=8,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      # batch efectivo = 32
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    dataloader_num_workers=2,
    report_to="none",
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

In [ ]:
# ── Guardar checkpoint final (entregable obligatorio) ─────────────────────────
model.save_pretrained(str(BASE / "decoder_checkpoint"))
tokenizer.save_pretrained(str(BASE / "decoder_checkpoint"))
print("Checkpoint guardado en decoder_checkpoint/")

---
## Sección 4 — Inference (RAG pipeline)

In [ ]:
# ── Cargar modelo si se está ejecutando desde un checkpoint ──────────────────
# (Omitir si ya está cargado en memoria desde la sección de training)

# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
# import torch
#
# MODEL_ID   = "meta-llama/Llama-3.2-1B-Instruct"
# CKPT_PATH  = str(BASE / "decoder_checkpoint")
#
# tokenizer = AutoTokenizer.from_pretrained(CKPT_PATH)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
# )
# model = PeftModel.from_pretrained(base_model, CKPT_PATH)
# model.eval()

In [ ]:
# ── Post-procesado y normalización de formato ─────────────────────────────────
def normalize_tool_call(raw: str) -> str:
    """Corrige errores de formato comunes y valida contra TOOLS_DEF."""
    raw = raw.strip()

    # 1. ¿Es no_action directamente?
    if raw.lower().startswith("no_action"):
        return "no_action"

    # 2. Extraer tool_name(params)
    m = re.match(r"^(\w+)\s*\((.*)\)$", raw, re.DOTALL)
    if not m:
        # Intentar extraer si hay texto antes/después
        m = re.search(r"(\w+)\s*\(([^)]*)\)", raw)
        if not m:
            return "no_action"
    name, params_str = m.group(1).strip(), m.group(2).strip()

    # 3. Corregir nombre de herramienta
    if name not in VALID_TOOLS:
        candidates = get_close_matches(name, VALID_TOOLS, n=1, cutoff=0.6)
        if not candidates:
            return "no_action"
        name = candidates[0]

    if name == "no_action":
        return "no_action"

    # 4. Parsear pares key=value
    parsed = {}
    for key, val in re.findall(r"(\w+)\s*=\s*('[^']*'|\d+)", params_str):
        parsed[key] = val

    # 5. Reordenar según TOOLS_DEF (orden de parámetros es obligatorio)
    expected = list(TOOLS_DEF[name].get("parameters", {}).keys())
    parts = []
    for key in expected:
        if key in parsed:
            parts.append(f"{key}={parsed[key]}")

    if not parts and expected:
        return "no_action"

    return f"{name}({','.join(parts)})"


# Test del normalizador
examples = [
    ("get_telemetry(module='condor',metric='temperature',timeframe_hours=6)",
     "get_telemetry(module='condor',metric='temperature',timeframe_hours=6)"),
    ("get_telemetry( module='condor' , metric='temperature', timeframe_hours=6)",
     "get_telemetry(module='condor',metric='temperature',timeframe_hours=6)"),
    ("no_action", "no_action"),
]
for raw, expected in examples:
    got = normalize_tool_call(raw)
    status = "OK" if got == expected else f"FAIL (got '{got}')"
    print(f"{status}: '{raw[:60]}'")

In [ ]:
# ── Función de predicción completa ────────────────────────────────────────────
model.eval()

def predict(query: str) -> str:
    top_chunks = retrieve(query, k=RETRIEVAL_K)
    prompt     = make_prompt(query, top_chunks[:PROMPT_K])
    inputs     = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKS,
            do_sample=False,           # greedy — determinista
            repetition_penalty=1.05,
        )

    raw = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    return normalize_tool_call(raw)

In [ ]:
# ── Evaluar en validation set ─────────────────────────────────────────────────
print("Evaluando en validation set...")
val_preds = [predict(q) for q in val_df["query"].tolist()]
val_gts   = val_df["tool_call"].tolist()

exact_match = sum(p == g for p, g in zip(val_preds, val_gts)) / len(val_gts)
print(f"\nExact Match (validation): {exact_match:.4f} ({exact_match*100:.2f}%)")

# Análisis de errores por herramienta
errors = [
    {"query": q, "gt": g, "pred": p, "tool": extract_tool_name(g)}
    for q, g, p in zip(val_df["query"], val_gts, val_preds)
    if p != g
]
print(f"\nErrores totales: {len(errors)}")
if errors:
    err_df = pd.DataFrame(errors)
    print("\nErrores por herramienta:")
    print(err_df["tool"].value_counts().to_string())
    print("\nPrimeros 5 errores:")
    for e in errors[:5]:
        print(f"  Query: {e['query'][:80]}")
        print(f"  GT:    {e['gt']}")
        print(f"  Pred:  {e['pred']}")
        print()

---
## Sección 5 — Answer File Generation

In [ ]:
print(f"Generando predicciones para {len(test_df)} queries de test...")

test_preds = []
for i, row in test_df.iterrows():
    pred = predict(row["query"])
    test_preds.append(pred)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(test_df)} procesadas...")

print("\nListo.")

In [ ]:
# ── Validar que todas las predicciones tienen formato válido ──────────────────
invalid = []
for i, pred in enumerate(test_preds):
    if pred == "no_action":
        continue
    m = re.match(r"^(\w+)\(.*\)$", pred)
    if not m or m.group(1) not in VALID_TOOLS:
        invalid.append((i, pred))

if invalid:
    print(f"ADVERTENCIA: {len(invalid)} predicciones con formato inválido:")
    for idx, pred in invalid[:10]:
        print(f"  [{idx}] {pred}")
else:
    print(f"OK — todas las {len(test_preds)} predicciones tienen formato válido.")

In [ ]:
# ── Guardar submission.csv ────────────────────────────────────────────────────
submission = pd.DataFrame({
    "id":        test_df["id"].tolist(),
    "tool_call": test_preds,
})

submission.to_csv(BASE / "submission.csv", index=False)
print("submission.csv guardado.")
submission.head(10)

In [ ]:
# ── Resumen final ─────────────────────────────────────────────────────────────
print("=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"Exact Match (validation):  {exact_match*100:.2f}%")
for k in [1, 3, 5]:
    m = retrieval_metrics(k)
    print(f"P@{k} / R@{k}:              {m[f'P@{k}']}")
print(f"Chunks en retrieval index: {len(chunks)}")
print(f"Test predictions:          {len(test_preds)}")
print("\nEntregables:")
print(f"  decoder_checkpoint/  — {'OK' if (BASE / 'decoder_checkpoint').exists() else 'FALTA'}")
print(f"  retrieval_index.json — {'OK' if (BASE / 'retrieval_index.json').exists() else 'FALTA'}")
print(f"  submission.csv       — {'OK' if (BASE / 'submission.csv').exists() else 'FALTA'}")
print(f"  requirements.txt     — {'OK' if (BASE / 'requirements.txt').exists() else 'FALTA'}")